# GROUP41 - Kaggle Submission Notebook
**CSIRO - Image2Biomass Prediction.** Kaggle'da internet kapalı çalışır: tüm eğitim verisiyle modeli eğitir, test görüntüleri
için tahmin üretir ve `submission.csv` yazar.

**Model:** `MODEL = "b3"`: ResNet50 özellikleri + Ridge (CV 0.531). `"b2"`: renk/örtü özellikleri + Ridge (CV 0.381).

**Input'lar:** yarışma verisi (`csiro-biomass`) ve ImageNet ResNet50 ağırlığını içeren dataset
(`resnet50-11ad3fa6.pth`, torchvision `IMAGENET1K_V2`). Internet kapalı olmalı.

İkisi de gram ölçeğinde, 5 hedef direkt ve CV'de seçilen alpha ile (bkz. `assignment1_eda_baseline.ipynb` Bölüm 10-13).
Fonksiyonlar `src/models.py` ile aynı. Kaggle'da repo klonlanamadığı için buraya kopyalandı.

In [ ]:
import os
import numpy as np, pandas as pd
from PIL import Image
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

ROOT = "/kaggle/input/competitions/csiro-biomass"
WEIGHTS_PATH = "/kaggle/input/datasets/omernuriz/resnet50-imagenet/resnet50-11ad3fa6.pth"

MODEL = "b3"   # "b2" yaparsan ağırlık gerekmez
ALPHA = {"b2": 100, "b3": 1000}[MODEL]   # GroupKFold CV'de seçilen değerler
TARGETS = ["Dry_Green_g", "Dry_Dead_g", "Dry_Clover_g", "GDM_g", "Dry_Total_g"]
print("Veri:", ROOT, "| Model:", MODEL, "| alpha:", ALPHA, "| ağırlık:", WEIGHTS_PATH)

In [ ]:
# train.csv: her görüntü tek satır (src/data.py::load_train ile aynı)
raw = pd.read_csv(f"{ROOT}/train.csv")
train = raw.pivot(index="image_path", columns="target_name", values="target")[TARGETS].reset_index()

test = pd.read_csv(f"{ROOT}/test.csv")
test_imgs = test.drop_duplicates("image_path")[["image_path"]].reset_index(drop=True)
print("train:", train.shape, "| test satır:", len(test), "| test görüntü:", len(test_imgs))

In [ ]:
# Özellikler (src/models.py ile aynı)
def color_features(path, size=(200, 100)):
    im = Image.open(path).convert("RGB").resize(size)
    rgb = np.asarray(im, dtype=float) / 255
    r, g, b = rgb[..., 0], rgb[..., 1], rgb[..., 2]
    toplam = r.mean() + g.mean() + b.mean() + 1e-9
    hsv = np.asarray(im.convert("HSV"), dtype=float).reshape(-1, 3) / 255
    return [r.mean(), g.mean(), b.mean(), r.std(), g.std(), b.std(),
            g.mean() / toplam, (2 * g - r - b).mean(), ((g > r) & (g > b)).mean(),
            *hsv.mean(0), *hsv.std(0)]


def cnn_features(paths, batch_size=16, tile=448):
    import torch, torchvision
    from torchvision import transforms as T
    device = "cuda" if torch.cuda.is_available() else "cpu"
    net = torchvision.models.resnet50(weights=None)
    net.load_state_dict(torch.load(WEIGHTS_PATH, map_location="cpu"))
    net.fc = torch.nn.Identity()
    net = net.eval().to(device)
    prep = T.Compose([T.Resize((tile, tile)), T.ToTensor(),
                      T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
    feats = []
    with torch.no_grad():
        for i in range(0, len(paths), batch_size):
            halves = []
            for p in paths[i:i + batch_size]:
                im = Image.open(p).convert("RGB")
                w, h = im.size
                halves += [prep(im.crop((0, 0, w // 2, h))), prep(im.crop((w // 2, 0, w, h)))]
            out = net(torch.stack(halves).to(device)).cpu().numpy()
            feats.append(out.reshape(-1, 2, out.shape[-1]).mean(1))
    return np.vstack(feats)


def features(df):
    paths = [f"{ROOT}/{p}" for p in df["image_path"]]
    if MODEL == "b3":
        return cnn_features(paths)
    return np.array([color_features(p) for p in paths])


X_train = features(train)
X_test = features(test_imgs)
print(X_train.shape, X_test.shape)

In [ ]:
# Eğitim: gram ölçeğinde StandardScaler + Ridge, tüm train verisiyle
Y = train[TARGETS].to_numpy()
model = make_pipeline(StandardScaler(), Ridge(alpha=ALPHA)).fit(X_train, Y)

pred = np.clip(model.predict(X_test), 0, None)
pred_df = pd.DataFrame(pred, columns=TARGETS)
pred_df.insert(0, "image_path", test_imgs["image_path"])
pred_df.round(1)

In [ ]:
# submission: sample_id, target (test.csv'deki satır sırasıyla)
long = pred_df.melt(id_vars="image_path", var_name="target_name", value_name="target")
sub = test.merge(long, on=["image_path", "target_name"], how="left")[["sample_id", "target"]]

assert len(sub) == len(test) and sub["target"].notna().all() and (sub["target"] >= 0).all()
sub.to_csv("submission.csv", index=False)
sub.head(10)